In [1]:
"""
先安装shap
pip install shap
pip install --upgrade typing_extensions
pip install plotly
pip install squarify
pip install openpyxl
"""
import pandas as pd
import numpy as np
# Plots
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.offline as py
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.tools as tls
import plotly.figure_factory as ff
py.init_notebook_mode(connected=True)
import squarify
import pickle

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import shap

In [2]:
## import data
df = pd.read_excel('../data/GD数据1230-updated.xlsx')

In [3]:
df

,Group01,Hos_name,Name,Age,WBC,NEUT_No,LYM_No,MON_No,EOS_No,BASO_No,...,MCH,MCHC,PLT,MPV,PDW,Plateletcrit_,CA125,NLR,PLR,SII
0,1,广医一院,邢嘉艺,20,4.40,2.00,1.80,NaN,NaN,NaN,...,NaN,NaN,189.0,NaN,NaN,NaN,28.70,1.111111,105.000000,210.000000
1,1,广医一院,魏粤,38,7.30,5.10,1.90,0.20,0.00,0.00,...,30.5,349.0,333.0,7.5,16.6,NaN,178.00,2.684211,175.263158,893.842105
2,1,广医一院,梁爱莲,36,5.80,3.90,1.30,0.40,0.20,0.00,...,30.1,344.0,208.0,9.2,17.0,NaN,33.20,3.000000,160.000000,624.000000
3,1,广医一院,张小莹,36,7.70,4.00,2.60,0.60,0.50,0.00,...,28.8,340.0,323.0,8.0,16.0,NaN,42.40,1.538462,124.230769,496.923077
4,1,广医一院,伍颖琳,25,8.10,5.20,2.10,0.60,0.10,0.10,...,32.0,348.0,375.0,8.4,16.2,31.4,35.70,2.476190,178.571429,928.571429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4600,0,广医一院,孙小双,35,10.57,7.60,2.20,0.60,0.14,0.04,...,25.4,334.0,349.0,8.3,16.5,29.1,14.40,3.454545,158.636364,1205.636364
4601,0,广医一院,孙小红,35,6.07,4.20,1.40,0.30,0.17,0.01,...,28.3,327.0,249.0,7.9,15.9,19.6,10.52,3.000000,177.857143,747.000000
4602,0,广医一院,孙如意,26,6.34,3.90,1.90,0.40,0.05,0.03,...,17.4,289.0,319.0,8.8,17.3,28.2,NaN,2.052632,167.894737,654.789474
4603,0,广医一院,孙榕,28,6.19,2.40,2.90,0.50,0.31,0.05,...,28.5,331.0,332.0,8.7,16.7,28.7,8.68,0.827586,114.482759,274.758621


In [4]:
df.columns

Index(['Group01', 'Hos_name', 'Name', 'Age', 'WBC', 'NEUT_No', 'LYM_No',
       'MON_No', 'EOS_No', 'BASO_No', 'NRBC_No', 'RBC', 'HGB', 'HCT', 'MCV',
       'MCH', 'MCHC', 'PLT', 'MPV', 'PDW', 'Plateletcrit_', 'CA125', 'NLR',
       'PLR', 'SII'],
      dtype='object')

### extract features for modeling

In [5]:
df_model = df.drop(columns=['Hos_name','Name']) ## delete useless features
print(df_model.shape)

(4605, 23)


### split train and test

In [10]:
from sklearn.model_selection import train_test_split

df_model_train, df_model_test = train_test_split(
    df_model,
    test_size=0.2, # 20%
    random_state=2025,
    stratify=df_model["Group01"]   # 分层变量
)

print(df_model_train.shape, df_model_test.shape)

df_model_train.to_pickle('../results/df_model_train.pkl')  # 保存
df_model_test.to_pickle('../results/df_model_test.pkl')

(3684, 23) (921, 23)


In [11]:
df_model_train.head()

,Group01,Age,WBC,NEUT_No,LYM_No,MON_No,EOS_No,BASO_No,NRBC_No,RBC,...,MCH,MCHC,PLT,MPV,PDW,Plateletcrit_,CA125,NLR,PLR,SII
3910,0,34,5.60,3.35,1.64,0.38,0.20,0.03,0.008,4.32,...,29.6,346.0,259.0,7.7,16.5,20.0,NaN,2.042683,157.926829,529.054878
4568,0,35,5.82,3.90,1.54,0.28,0.05,0.05,0.005,3.80,...,30.3,338.0,189.0,8.7,16.2,16.4,NaN,2.532468,122.727273,478.636364
830,1,55,5.60,4.30,1.00,0.20,0.00,0.00,0.000,4.49,...,19.7,304.0,361.0,6.7,16.0,24.3,17.85,4.300000,361.000000,1552.300000
4294,0,35,6.90,3.60,2.70,0.50,0.09,0.02,0.010,4.92,...,27.2,321.0,211.0,8.6,16.0,18.1,NaN,1.333333,78.148148,281.333333
2787,0,24,7.70,4.80,2.30,0.50,0.10,0.00,0.010,4.34,...,31.2,334.0,366.0,7.7,15.7,28.2,NaN,2.086957,159.130435,763.826087


### fill NA with KNN

In [12]:
from sklearn.impute import KNNImputer

# 1. 初始化 KNN 填充器（默认n_neighbors=5）
imputer = KNNImputer(n_neighbors=5)

# 2. 仅填充包含缺失值的列
cols_with_nan = df_model.columns[df_model.isnull().any()]
train_to_fill = df_model_train[cols_with_nan]
test_to_fill = df_model_test[cols_with_nan]

# 3. 应用KNN填充
train_filled_array = imputer.fit_transform(train_to_fill)
test_filled_array = imputer.transform(test_to_fill) # 用training data的fit result来填充test

# 4. 构建填充后的数据框
train_df_filled = pd.DataFrame(train_filled_array, columns=cols_with_nan, index=df_model_train.index)
test_df_filled = pd.DataFrame(test_filled_array, columns=cols_with_nan, index=df_model_test.index)

# 5. 替换原来的列，得到df_model3
df_model_knn_train = df_model_train.copy()
df_model_knn_train[cols_with_nan] = train_df_filled

df_model_knn_test = df_model_test.copy()
df_model_knn_test[cols_with_nan] = test_df_filled

df_model_knn_train.to_pickle('../results/df_model_knn_train.pkl')  # 保存
df_model_knn_test.to_pickle('../results/df_model_knn_test.pkl')

## 建模-XGBoost

In [2]:
#pip install --upgrade xgboost==2.1.4
import xgboost as xgb

# Data processing, metrics and modeling
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV,StratifiedKFold
from sklearn.metrics import precision_score, recall_score, confusion_matrix,  roc_curve, precision_recall_curve, accuracy_score, roc_auc_score,f1_score
import lightgbm as lgbm
from sklearn.ensemble import VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_curve,auc
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_predict
# from yellowbrick.classifier import DiscriminationThreshold

import plotly.offline as pyo

In [3]:
# 可视化最后模型指标报告
def model_performance(model, X, y, filename) :   
    # StratifiedKfold
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state = 42) #固定seed
    
    all_results = []
    fold_auc_list = []
    fold = 1
    
    for train,test in cv.split(X,y):
        ## model training and prediction
        model.fit(X.iloc[train], y.iloc[train])
        pred_proba = model.predict_proba(X.iloc[test])
        y_pred_binary = (pred_proba[:,1] >= 0.5).astype(int)
        
        ## fold: auc result
        #print(y_pred_binary)
        Accuracy = accuracy_score(y.iloc[test], y_pred_binary)
        Precision = precision_score(y.iloc[test], y_pred_binary)
        Recall = recall_score(y.iloc[test], y_pred_binary)
        F1 = f1_score(y.iloc[test], y_pred_binary)
        #precision, recall, _ = precision_recall_curve(y.iloc[test], pred_proba[:,1])
        fpr, tpr, t = roc_curve(y.iloc[test], pred_proba[:, 1])
        roc_auc = auc(fpr, tpr)
        fold_auc_list.append([fold, roc_auc, Accuracy, Precision, Recall, F1])
        
        ## save prediction result
        fold_df = pd.DataFrame({
            "fold": fold,
            "id": test,
            "y_true": y.iloc[test].values,
            "y_proba": pred_proba[:,1]
        })
        all_results.append(fold_df)
        fold +=1
        
    # 合并所有样本结果
    all_results_df = pd.concat(all_results, axis=0)

    # 保存结果文件
    all_results_filename = f"{filename}_cv_predictions.pkl"
    auc_summary_filename = f"{filename}_cv_auc.csv"

    all_results_df.to_pickle(all_results_filename)
    pd.DataFrame(fold_auc_list, columns=["fold", "AUC", "accuracy", "precision", "recall", "F1"]).to_csv(auc_summary_filename, index=False)

    print(f"✓ 每折样本预测结果已保存到：{all_results_filename}")
    print(f"✓ 每折 AUC 已保存到：{auc_summary_filename}")

    return pd.DataFrame(fold_auc_list, columns=["fold", "AUC", "accuracy", "precision", "recall", "F1"])

#### data processing

In [4]:
# Def X and Y
df_model_knn_train = pickle.load(open('../results/df_model_knn_train.pkl','rb'))
df_model_knn_test = pickle.load(open('../results/df_model_knn_test.pkl','rb'))

X = df_model_knn_train.drop('Group01', axis=1)
y = df_model_knn_train['Group01']

"""
### for with NaN ###
df_model2_train = pickle.load(open('../results/df_model_train.pkl','rb'))
df_model2_test = pickle.load(open('../results/df_model_test.pkl','rb'))

X = df_model2_train.drop('Group', axis=1)
y = df_model2_train['Group']
"""

"\n### for with NaN ###\ndf_model2_train = pickle.load(open('../results/df_model_train.pkl','rb'))\ndf_model2_test = pickle.load(open('../results/df_model_test.pkl','rb'))\n\nX = df_model2_train.drop('Group', axis=1)\ny = df_model2_train['Group']\n"

In [5]:
X

,Age,WBC,NEUT_No,LYM_No,MON_No,EOS_No,BASO_No,NRBC_No,RBC,HGB,...,MCH,MCHC,PLT,MPV,PDW,Plateletcrit_,CA125,NLR,PLR,SII
3910,34,5.60,3.35,1.64,0.38,0.20,0.03,0.0080,4.32,128.0,...,29.6,346.0,259.0,7.7,16.5,20.0,16.084,2.042683,157.926829,529.054878
4568,35,5.82,3.90,1.54,0.28,0.05,0.05,0.0050,3.80,115.0,...,30.3,338.0,189.0,8.7,16.2,16.4,41.710,2.532468,122.727273,478.636364
830,55,5.60,4.30,1.00,0.20,0.00,0.00,0.0000,4.49,88.0,...,19.7,304.0,361.0,6.7,16.0,24.3,17.850,4.300000,361.000000,1552.300000
4294,35,6.90,3.60,2.70,0.50,0.09,0.02,0.0100,4.92,134.0,...,27.2,321.0,211.0,8.6,16.0,18.1,63.606,1.333333,78.148148,281.333333
2787,24,7.70,4.80,2.30,0.50,0.10,0.00,0.0100,4.34,135.0,...,31.2,334.0,366.0,7.7,15.7,28.2,48.874,2.086957,159.130435,763.826087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4604,33,6.59,4.43,1.66,0.44,0.04,0.01,0.0050,4.35,112.0,...,25.9,328.0,207.0,9.9,17.2,20.3,16.260,2.668675,124.698795,552.415663
1379,38,6.28,3.33,2.51,0.30,0.08,0.02,0.0040,4.18,130.0,...,31.1,342.0,333.0,9.5,9.5,32.0,12.300,1.326693,132.669323,441.788845
3855,33,6.10,3.20,2.30,0.30,0.30,0.00,0.0100,5.08,132.0,...,25.9,320.0,242.0,8.8,18.0,21.2,16.320,1.391304,105.217391,336.695652
2102,35,4.13,2.14,1.52,0.30,0.06,0.10,0.0026,4.60,80.0,...,17.4,282.0,135.0,10.5,14.8,14.0,46.612,1.407895,88.815789,190.065789


In [6]:
y

3910    0
4568    0
830     1
4294    0
2787    0
       ..
4604    0
1379    1
3855    0
2102    1
902     1
Name: Group01, Length: 3684, dtype: int64

#### ten-fold cv

In [7]:
# 初始化 XGBoost 分类器，使用默认超参数
xgb_clf = xgb.XGBClassifier(random_state=2025, use_label_encoder=False, eval_metric='logloss')

In [8]:
model_performance(xgb_clf, X, y, "../results/xgboost/cv/XGBoost_model")

/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:28] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:29] WARNING: /workspace/src/l

✓ 每折样本预测结果已保存到：../results/xgboost/cv/XGBoost_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/xgboost/cv/XGBoost_model_cv_auc.csv


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.




,fold,AUC,accuracy,precision,recall,F1
0,1,0.961705,0.905149,0.901163,0.895954,0.898551
1,2,0.938259,0.872629,0.866279,0.861272,0.863768
2,3,0.956559,0.897019,0.894737,0.884393,0.889535
3,4,0.970273,0.913279,0.912281,0.901734,0.906977
4,5,0.951768,0.915761,0.937888,0.877907,0.906907
5,6,0.968943,0.921196,0.944099,0.883721,0.912913
6,7,0.952287,0.891304,0.907407,0.854651,0.880240
7,8,0.980037,0.932065,0.940120,0.912791,0.926254
8,9,0.971701,0.932065,0.929825,0.924419,0.927114
9,10,0.969210,0.902174,0.930380,0.854651,0.890909


#### 重新训练模型并在test set测试

In [9]:
import joblib

# 模型已经在上面定义过了

X_test = df_model_knn_test.drop('Group01', axis=1)
y_test = df_model_knn_test['Group01']

# 3. 训练模型
xgb_clf.fit(X, y)

# 4. 预测
y_pred = xgb_clf.predict(X_test)
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(xgb_clf, '../results/xgboost/test/xgb_clf_model.pkl')

/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[01:43:35] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.




['../results/xgboost/test/xgb_clf_model.pkl']

#### 评估多个指标

In [10]:
# 6. 评估多个指标
metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "XGBoost"
}

print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/xgboost/test/xgb_clf_model.csv', index=False)

{'Accuracy': 0.9022801302931596, 'Precision': 0.9049881235154394, 'Recall': 0.8839907192575406, 'F1': 0.8943661971830986, 'AUC': 0.9623609072399262, 'Model': 'XGBoost'}


In [11]:
import plotly.offline as pyo
def generate_report(y, y_pred, y_proba, filename, subtitle):    
    conf_matrix = confusion_matrix(y, y_pred)
    trace1 = go.Heatmap(z = conf_matrix  ,x = ["0 (pred)","1 (pred)"],
                        y = ["0 (true)","1 (true)"],xgap = 2, ygap = 2, 
                        colorscale = 'Viridis', showscale  = False)
    
    #Show metrics
    tp = conf_matrix[1,1]
    fn = conf_matrix[1,0]
    fp = conf_matrix[0,1]
    tn = conf_matrix[0,0]
    Accuracy  =  ((tp+tn)/(tp+tn+fp+fn))
    Precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    Recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    F1_score  =  (2*(((tp/(tp+fp))*(tp/(tp+fn)))/((tp/(tp+fp))+(tp/(tp+fn)))))

    show_metrics = pd.DataFrame(data=[[Accuracy , Precision, Recall, F1_score]])
    show_metrics = show_metrics.T

    colors = ['gold', 'lightgreen', 'lightcoral', 'lightskyblue']
    trace2 = go.Bar(x = (show_metrics[0].values), 
                    y = ['Accuracy', 'Precision', 'Recall', 'F1_score'], text = np.round_(show_metrics[0].values,4),
                    textposition = 'auto', textfont=dict(color='black'),
                    orientation = 'h', opacity = 1, marker=dict(
            color=colors,
            line=dict(color='#000000',width=1.5)))

    #Roc curve
    fpr, tpr, _ = roc_curve(y, y_proba)
    auc_value = auc(fpr, tpr)

    trace3 = go.Scatter(x=fpr, y=tpr,
                        name = "Roc : " ,
                        line = dict(color = ('rgb(22, 96, 167)'),width = 2), fill='tozeroy')
    trace4 = go.Scatter(x = [0,1],y = [0,1],
                        line = dict(color = ('black'),width = 1.5,
                        dash = 'dot'))
    
    #Precision - recall curve
    precision, recall, _ = precision_recall_curve(y, y_proba)

    trace5 = go.Scatter(x = recall, y = precision,
                        name = "Precision" + str(precision),
                        line = dict(color = ('lightcoral'),width = 2), fill='tozeroy')
    
    auc_value=round(auc_value,3)
    #Subplots
    fig = tls.make_subplots(rows=2, cols=2, print_grid=False,
                          specs=[[{}, {}], 
                                 [{}, {}]],
                          subplot_titles=('Confusion Matrix',
                                          'Metrics',
                                          'ROC curve'+" "+ '('+ str(auc_value)+')',
                                          'Precision - Recall curve',
                                          ))
    #Trace and layout
    fig.append_trace(trace1,1,1)
    fig.append_trace(trace2,1,2)
    fig.append_trace(trace3,2,1)
    fig.append_trace(trace4,2,1)
    fig.append_trace(trace5,2,2)
    
    fig['layout'].update(showlegend = False, title = '<b>Model performance report (test)</b><br>'+subtitle,
                        autosize = False, height = 830, width = 830,
                        plot_bgcolor = 'white',
                        paper_bgcolor = 'white',
                        margin = dict(b = 195), font=dict(color='black'))
    fig["layout"]["xaxis1"].update(color = 'black')
    fig["layout"]["yaxis1"].update(color = 'black')
    fig["layout"]["xaxis2"].update((dict(range=[0, 1], color = 'black')))
    fig["layout"]["yaxis2"].update(color = 'black')
    fig["layout"]["xaxis3"].update(dict(title = "false positive rate"), color = 'black')
    fig["layout"]["yaxis3"].update(dict(title = "true positive rate"),color = 'black')
    fig["layout"]["xaxis4"].update(dict(title = "recall"), range = [0,1.05],color = 'black')
    fig["layout"]["yaxis4"].update(dict(title = "precision"), range = [0,1.05],color = 'black')
    for i in fig['layout']['annotations']:
        i['font'] = titlefont=dict(color='black', size = 14)
    pyo.plot(fig, filename=filename, auto_open=True)

In [12]:
generate_report(y_test, y_pred, y_proba, '../results/xgboost/test/report_model.html', "XGboost")

/opt/conda/lib/python3.9/site-packages/plotly/tools.py:453: DeprecationWarning:

plotly.tools.make_subplots is deprecated, please use plotly.subplots.make_subplots instead



#### 特征贡献度

##### 全局特征贡献度

In [14]:
# 2. 初始化 SHAP解释器
explainer = shap.TreeExplainer(xgb_clf, X)

# 3. 计算每个样本的 SHAP值
shap_values = explainer(X)

# 4. 全局特征重要性图（可看最重要的分类列）
shap.plots.bar(shap_values, max_display=10, show=False)  # 展示Top 10特征
plt.tight_layout()   # 避免标签被裁剪
plt.savefig("../results/figs/XBG_shap1.png", dpi=300, bbox_inches='tight')
plt.close()          # 可选：避免在循环中重复显示

In [15]:
# 绘制 summary_plot
shap.summary_plot(shap_values, X, show=False)

# 保存为 HTML 图像（使用 shap's save_html 不支持 summary_plot，需用matplotlib）
plt.savefig("../results/figs/XBG_shap2.png", dpi=300, bbox_inches='tight')
plt.close()

## random forest
##### 之后所有的模型全都用knn过的数据

#### 十折交叉验证

In [16]:
from sklearn.ensemble import RandomForestClassifier

# 初始化一个随机森林分类器，设置随机种子保证可重复
rf_clf = RandomForestClassifier(random_state=2025, n_estimators=100, max_depth=None)

In [17]:
model_performance(rf_clf, X, y, "../results/randomforest/cv/rf_model")

✓ 每折样本预测结果已保存到：../results/randomforest/cv/rf_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/randomforest/cv/rf_model_cv_auc.csv


,fold,AUC,accuracy,precision,recall,F1
0,1,0.949348,0.897019,0.919255,0.855491,0.886228
1,2,0.916878,0.856369,0.857143,0.832370,0.844575
2,3,0.944231,0.861789,0.876543,0.820809,0.847761
3,4,0.957547,0.907859,0.906433,0.895954,0.901163
4,5,0.951071,0.902174,0.914634,0.872093,0.892857
5,6,0.964434,0.899457,0.929936,0.848837,0.887538
6,7,0.940674,0.861413,0.849711,0.854651,0.852174
7,8,0.971983,0.918478,0.927711,0.895349,0.911243
8,9,0.974712,0.923913,0.900000,0.941860,0.920455
9,10,0.971405,0.913043,0.948718,0.860465,0.902439


#### 重新训练并在test set上测试

In [18]:
# 模型已经在上面定义过了
# 2. 划分训练/测试集，分层分割
X_test = df_model_knn_test.drop('Group01', axis=1)
y_test = df_model_knn_test['Group01']

# 3. 训练模型
rf_clf.fit(X, y)

# 4. 预测
y_pred = rf_clf.predict(X_test)
y_proba = rf_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(rf_clf, '../results/randomforest/test/rf_clf.pkl')

['../results/randomforest/test/rf_clf.pkl']

In [19]:
metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "RF"
}

print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/randomforest/test/rf_clf_model.csv', index=False)

{'Accuracy': 0.8892508143322475, 'Precision': 0.9041769041769042, 'Recall': 0.8538283062645011, 'F1': 0.8782816229116945, 'AUC': 0.9444528623514372, 'Model': 'RF'}


#### 特征贡献度

In [20]:
# 创建KernelExplainer，传入模型的预测函数和背景数据
explainer = shap.TreeExplainer(rf_clf)

# 计算SHAP值（分类模型predict_proba输出多类概率，shap_values是列表）
shap_values = explainer.shap_values(X)

# 绘制summary plot，假设二分类，看类别1的SHAP值
shap.summary_plot(shap_values[:, :, 1], X, show=False)

# 保存为图片（注意：不能保存为HTML，summary_plot是Matplotlib图）
plt.savefig("../results/figs/rf_shap2.png", dpi=300, bbox_inches='tight')
plt.close()

## 逻辑回归

#### 十折交叉验证

In [22]:
from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression(random_state=42, max_iter=1000)

model_performance(lr_clf, X, y, "../results/lr/cv/lr_clf_model")

/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

/opt/conda/lib/python3.9/site-packages/sklearn/lin

✓ 每折样本预测结果已保存到：../results/lr/cv/lr_clf_model3_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/lr/cv/lr_clf_model3_cv_auc.csv


/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



,fold,AUC,accuracy,precision,recall,F1
0,1,0.889362,0.810298,0.845638,0.728324,0.782609
1,2,0.844963,0.766938,0.766871,0.722543,0.744048
2,3,0.857467,0.785908,0.809211,0.710983,0.756923
3,4,0.872095,0.777778,0.779141,0.734104,0.755952
4,5,0.864440,0.798913,0.831081,0.715116,0.768750
5,6,0.847473,0.752717,0.795620,0.633721,0.705502
6,7,0.852085,0.771739,0.789474,0.697674,0.740741
7,8,0.870877,0.801630,0.807453,0.755814,0.780781
8,9,0.913117,0.834239,0.849057,0.784884,0.815710
9,10,0.907511,0.812500,0.836601,0.744186,0.787692


#### 重新训练并在test set上测试

In [23]:
# 模型已经在上面定义过了
# 3. 训练模型
lr_clf.fit(X, y)

# 4. 预测
y_pred = lr_clf.predict(X_test)
y_proba = lr_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(lr_clf, '../results/lr/test/lr_clf.pkl')

metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "LogisticRegression"
}
print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/lr/test/lr_clf_model.csv', index=False)

{'Accuracy': 0.7980456026058632, 'Precision': 0.8132992327365729, 'Recall': 0.7378190255220418, 'F1': 0.7737226277372263, 'AUC': 0.8659500923339173, 'Model': 'LogisticRegression'}


/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



## SVM

#### 十折交叉验证

In [24]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

svm = SVC(kernel='rbf', probability=True, random_state=2025)
svm_clf = CalibratedClassifierCV(svm)  # 概率校准，输出 predict_proba

model_performance(svm_clf, X, y, "../results/svm/cv/svm_clf_model")

✓ 每折样本预测结果已保存到：../results/svm/cv/svm_clf_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/svm/cv/svm_clf_model_cv_auc.csv


,fold,AUC,accuracy,precision,recall,F1
0,1,0.756296,0.715447,0.793103,0.531792,0.636678
1,2,0.730624,0.680217,0.682119,0.595376,0.635802
2,3,0.727793,0.661247,0.693548,0.497110,0.579125
3,4,0.730683,0.680217,0.700730,0.554913,0.619355
4,5,0.754835,0.687500,0.739496,0.511628,0.604811
5,6,0.700670,0.633152,0.658120,0.447674,0.532872
6,7,0.748858,0.676630,0.726496,0.494186,0.588235
7,8,0.778921,0.714674,0.763780,0.563953,0.648829
8,9,0.808466,0.725543,0.759124,0.604651,0.673139
9,10,0.757920,0.673913,0.716667,0.500000,0.589041


#### 重新训练并在test set上测试

In [25]:
# 3. 训练模型
svm_clf.fit(X, y)

# 4. 预测
y_pred = svm_clf.predict(X_test)
y_proba = svm_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(svm_clf, '../results/svm/test/svm_clf.pkl')

metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "SVM"
}
print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/svm/test/svm_clf_model.csv', index=False)

{'Accuracy': 0.6666666666666666, 'Precision': 0.6856287425149701, 'Recall': 0.531322505800464, 'F1': 0.5986928104575163, 'AUC': 0.7332828258913775, 'Model': 'SVM'}


## lightGBM

#### 十折交叉验证

In [26]:
from lightgbm import LGBMClassifier

lgbm_clf = LGBMClassifier(random_state=2025)
model_performance(lgbm_clf, X, y, "../results/lgbm/cv/lgbm_clf_model")

✓ 每折样本预测结果已保存到：../results/lgbm/cv/lgbm_clf_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/lgbm/cv/lgbm_clf_model_cv_auc.csv


,fold,AUC,accuracy,precision,recall,F1
0,1,0.964994,0.907859,0.911243,0.890173,0.900585
1,2,0.935959,0.864499,0.868263,0.838150,0.852941
2,3,0.956411,0.902439,0.920245,0.867052,0.892857
3,4,0.968768,0.910569,0.902299,0.907514,0.904899
4,5,0.954200,0.913043,0.926829,0.883721,0.904762
5,6,0.970011,0.904891,0.930818,0.860465,0.894260
6,7,0.956410,0.891304,0.907407,0.854651,0.880240
7,8,0.978465,0.948370,0.958084,0.930233,0.943953
8,9,0.973096,0.932065,0.934911,0.918605,0.926686
9,10,0.970307,0.915761,0.937888,0.877907,0.906907


#### 重新训练并在test set上测试

In [27]:
# 3. 训练模型
lgbm_clf.fit(X, y)

# 4. 预测
y_pred = lgbm_clf.predict(X_test)
y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(lgbm_clf, '../results/lgbm/test/lgbm_clf.pkl')

metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "LightGBM"
}
print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/lgbm/test/lgbm_clf_model.csv', index=False)

{'Accuracy': 0.9033659066232356, 'Precision': 0.9232673267326733, 'Recall': 0.8654292343387471, 'F1': 0.8934131736526947, 'AUC': 0.9669184146976656, 'Model': 'LightGBM'}


#### SHAP

In [28]:
# 创建解释器
explainer = shap.TreeExplainer(lgbm_clf)

# 计算SHAP值
shap_values = explainer.shap_values(X)

# 绘制总结图
shap.summary_plot(shap_values, X, show=False)

# 保存为 HTML 图像（使用 shap's save_html 不支持 summary_plot，需用matplotlib）
plt.savefig("../results/figs/lgbm_shap.png", dpi=300, bbox_inches='tight')
plt.close()

/opt/conda/lib/python3.9/site-packages/shap/explainers/_tree.py:586: UserWarning:

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray



## KNN

#### 十折交叉验证

In [29]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier(n_neighbors=5)
model_performance(knn_clf, X, y, "../results/knn/cv/knn_clf_model")

✓ 每折样本预测结果已保存到：../results/knn/cv/knn_clf_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/knn/cv/knn_clf_model_cv_auc.csv


,fold,AUC,accuracy,precision,recall,F1
0,1,0.668072,0.617886,0.612676,0.502890,0.552381
1,2,0.668589,0.623306,0.604938,0.566474,0.585075
2,3,0.705423,0.655827,0.664286,0.537572,0.594249
3,4,0.739295,0.699187,0.718310,0.589595,0.647619
4,5,0.691371,0.663043,0.696721,0.494186,0.578231
5,6,0.671630,0.646739,0.666667,0.488372,0.563758
6,7,0.680158,0.633152,0.629371,0.523256,0.571429
7,8,0.695761,0.668478,0.681159,0.546512,0.606452
8,9,0.717326,0.665761,0.662252,0.581395,0.619195
9,10,0.652053,0.622283,0.625954,0.476744,0.541254


#### 重新训练并在test set上测试

In [30]:
# 3. 训练模型
knn_clf.fit(X, y)

# 4. 预测
y_pred = knn_clf.predict(X_test)
y_proba = knn_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(knn_clf, '../results/knn/test/knn_clf.pkl')

metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "KNN"
}
print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/knn/test/knn_clf_model.csv', index=False)

{'Accuracy': 0.6753528773072747, 'Precision': 0.6952662721893491, 'Recall': 0.5452436194895591, 'F1': 0.6111833550065019, 'AUC': 0.7013944789052513, 'Model': 'KNN'}


## MLP

#### 十折交叉验证

In [31]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid

# 参数网格（你可以根据数据量微调）
param_grid = {
    'hidden_layer_sizes': [(100,), (100, 50), (128, 64, 32)],  # 层数与神经元个数
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],   # 正则化
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [300],  # 固定训练轮数
}
grid = list(ParameterGrid(param_grid))

print("参数组合数量：", len(grid))
for params in grid:
    print(params)

参数组合数量： 36
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate': 'constant', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate': 'adaptive', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate': 'constant', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate': 'adaptive', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate': 'constant', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate': 'adaptive', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (100,), 'learning_rate': 'constant', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (100,), 'learning_rate': 'adaptive', 'max_iter': 300}
{'activation': 'relu', 'alpha': 0.001, 'hid

In [32]:
for params in grid:
    print(params)
    mlp = MLPClassifier(random_state=2025, 
                        hidden_layer_sizes=params['hidden_layer_sizes'],
                        activation=params['activation'],
                        alpha=params['alpha'],
                        learning_rate=params['learning_rate'],
                        max_iter=params['max_iter']
                       )
    ofilename = f"../results/mlp/cv/mlp_{params['activation']}_{params['alpha']}_{params['learning_rate']}_{params['hidden_layer_sizes']}_model"
    print(ofilename)
    model_performance(mlp, X, y, ofilename)

{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate': 'constant', 'max_iter': 300}
../results/mlp/cv/mlp_relu_0.0001_constant_(100,)_model
✓ 每折样本预测结果已保存到：../results/mlp/cv/mlp_relu_0.0001_constant_(100,)_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/mlp/cv/mlp_relu_0.0001_constant_(100,)_model_cv_auc.csv
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate': 'adaptive', 'max_iter': 300}
../results/mlp/cv/mlp_relu_0.0001_adaptive_(100,)_model
✓ 每折样本预测结果已保存到：../results/mlp/cv/mlp_relu_0.0001_adaptive_(100,)_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/mlp/cv/mlp_relu_0.0001_adaptive_(100,)_model_cv_auc.csv
{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'learning_rate': 'constant', 'max_iter': 300}
../results/mlp/cv/mlp_relu_0.0001_constant_(100, 50)_model
✓ 每折样本预测结果已保存到：../results/mlp/cv/mlp_relu_0.0001_constant_(100, 50)_model_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/mlp/cv/mlp_relu_0.

In [33]:
max_auc = 0
max_params = None
for params in grid:
    ofilename = f"../results/mlp/cv/mlp_{params['activation']}_{params['alpha']}_{params['learning_rate']}_{params['hidden_layer_sizes']}_model"
    res = pd.read_csv(f'{ofilename}_cv_auc.csv')
    mean_auc = max(res.AUC)
    if mean_auc > max_auc:
        max_auc = mean_auc
        max_params = params
        print(f'{max_auc}:{max_params}')

print(f"best params:{max_params}")

0.8818521594684385:{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'learning_rate': 'constant', 'max_iter': 300}
0.8853820598006644:{'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate': 'constant', 'max_iter': 300}
0.8867465590887518:{'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate': 'constant', 'max_iter': 300}
best params:{'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate': 'constant', 'max_iter': 300}


#### 重新训练并在test set上测试

In [34]:
# 参数网格（你可以根据数据量微调）
param_grid = {
    'hidden_layer_sizes': (128, 64, 32),  # 层数与神经元个数
    'activation': 'relu',
    'alpha': 0.01,   # 正则化
    'learning_rate': 'constant',
    'max_iter': 300,  # 固定训练轮数
}

mlp = MLPClassifier(random_state=2025, 
                    hidden_layer_sizes=params['hidden_layer_sizes'],
                    activation=params['activation'],
                    alpha=params['alpha'],
                    learning_rate=params['learning_rate'],
                    max_iter=params['max_iter']
                   )

# 3. 训练模型
mlp.fit(X, y)

# 4. 预测
y_pred = mlp.predict(X_test)
y_proba = mlp.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(mlp, '../results/mlp/test/mlp.pkl')

metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "MLP_Tuned"
}
print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/mlp/test/mlp_model.csv', index=False)

{'Accuracy': 0.7774158523344191, 'Precision': 0.8424242424242424, 'Recall': 0.6450116009280742, 'F1': 0.7306176084099868, 'AUC': 0.8357782091955112, 'Model': 'MLP_Tuned'}


## XGBOOST-with NaN

#### 十折交叉验证

In [43]:
df_model_train = pickle.load(open('../results/df_model_train.pkl','rb'))
df_model_test = pickle.load(open('../results/df_model_test.pkl','rb'))

X = df_model_train.drop('Group01', axis=1)
y = df_model_train['Group01']

In [44]:
X

,Age,WBC,NEUT_No,LYM_No,MON_No,EOS_No,BASO_No,NRBC_No,RBC,HGB,...,MCH,MCHC,PLT,MPV,PDW,Plateletcrit_,CA125,NLR,PLR,SII
3910,34,5.60,3.35,1.64,0.38,0.20,0.03,0.008,4.32,128.0,...,29.6,346.0,259.0,7.7,16.5,20.0,NaN,2.042683,157.926829,529.054878
4568,35,5.82,3.90,1.54,0.28,0.05,0.05,0.005,3.80,115.0,...,30.3,338.0,189.0,8.7,16.2,16.4,NaN,2.532468,122.727273,478.636364
830,55,5.60,4.30,1.00,0.20,0.00,0.00,0.000,4.49,88.0,...,19.7,304.0,361.0,6.7,16.0,24.3,17.85,4.300000,361.000000,1552.300000
4294,35,6.90,3.60,2.70,0.50,0.09,0.02,0.010,4.92,134.0,...,27.2,321.0,211.0,8.6,16.0,18.1,NaN,1.333333,78.148148,281.333333
2787,24,7.70,4.80,2.30,0.50,0.10,0.00,0.010,4.34,135.0,...,31.2,334.0,366.0,7.7,15.7,28.2,NaN,2.086957,159.130435,763.826087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4604,33,6.59,4.43,1.66,0.44,0.04,0.01,0.005,4.35,112.0,...,25.9,328.0,207.0,9.9,17.2,20.3,16.26,2.668675,124.698795,552.415663
1379,38,6.28,3.33,2.51,0.30,0.08,0.02,NaN,4.18,130.0,...,31.1,342.0,333.0,9.5,9.5,32.0,12.30,1.326693,132.669323,441.788845
3855,33,6.10,3.20,2.30,0.30,0.30,0.00,0.010,5.08,132.0,...,25.9,320.0,242.0,8.8,18.0,21.2,NaN,1.391304,105.217391,336.695652
2102,35,4.13,2.14,1.52,0.30,0.06,0.10,NaN,4.60,80.0,...,17.4,282.0,135.0,10.5,14.8,14.0,NaN,1.407895,88.815789,190.065789


In [45]:
# 初始化 XGBoost 分类器，使用默认超参数
xgb_clf = xgb.XGBClassifier(random_state=2025, use_label_encoder=False, eval_metric='logloss')

model_performance(xgb_clf, X, y, "../results/xgboost/cv/XGBoost_model_withnan")

/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:47:46] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:47:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:47:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:47:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:47:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.


/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:47:47] WARNING: /workspace/src/l

✓ 每折样本预测结果已保存到：../results/xgboost/cv/XGBoost_model_withnan_cv_predictions.pkl
✓ 每折 AUC 已保存到：../results/xgboost/cv/XGBoost_model_withnan_cv_auc.csv


,fold,AUC,accuracy,precision,recall,F1
0,1,0.988248,0.953930,0.953488,0.947977,0.950725
1,2,0.977350,0.934959,0.940828,0.919075,0.929825
2,3,0.980919,0.959350,0.970238,0.942197,0.956012
3,4,0.988587,0.951220,0.953216,0.942197,0.947674
4,5,0.983774,0.953804,0.964072,0.936047,0.949853
5,6,0.988164,0.951087,0.958333,0.936047,0.947059
6,7,0.987883,0.934783,0.940476,0.918605,0.929412
7,8,0.991190,0.956522,0.953488,0.953488,0.953488
8,9,0.996678,0.975543,0.965714,0.982558,0.974063
9,10,0.981802,0.951087,0.969512,0.924419,0.946429


#### 重新训练模型并在test set测试

In [38]:
import joblib

# 模型已经在上面定义过了

X_test = df_model_test.drop('Group01', axis=1)
y_test = df_model_test['Group01']

# 3. 训练模型
xgb_clf.fit(X, y)

# 4. 预测
y_pred = xgb_clf.predict(X_test)
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

# 5. 保存训练好的模型
joblib.dump(xgb_clf, '../results/xgboost/test/xgb_clf_model_withnan.pkl')

/opt/conda/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning:

[02:25:23] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.




['../results/xgboost/test/xgb_clf_model_withnan.pkl']

In [39]:
metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "XGBoost"
}

print(metrics_dict)
pd.DataFrame.from_dict([metrics_dict]).to_csv('../results/xgboost/test/xgb_clf_model_withnan.csv', index=False)

{'Accuracy': 0.9543973941368078, 'Precision': 0.964200477326969, 'Recall': 0.9373549883990719, 'F1': 0.9505882352941176, 'AUC': 0.9901510488185994, 'Model': 'XGBoost'}


In [40]:
generate_report(y_test, y_pred, y_proba, '../results/xgboost/test/report_model_withnan.html', "XGboost with nan")

/opt/conda/lib/python3.9/site-packages/plotly/tools.py:453: DeprecationWarning:

plotly.tools.make_subplots is deprecated, please use plotly.subplots.make_subplots instead



#### SHAP

In [41]:
# 2. 初始化 SHAP解释器
explainer = shap.TreeExplainer(xgb_clf, X)

# 3. 计算每个样本的 SHAP值
shap_values = explainer(X)

# 4. 全局特征重要性图（可看最重要的分类列）
shap.plots.bar(shap_values, max_display=10, show=False)  # 展示Top 10特征
plt.tight_layout()   # 避免标签被裁剪
plt.savefig("../results/figs/XBG_withnan_shap1.png", dpi=300, bbox_inches='tight')
plt.close()          # 可选：避免在循环中重复显示

In [42]:
# 绘制 summary_plot
shap.summary_plot(shap_values, X, show=False)

# 保存为 HTML 图像（使用 shap's save_html 不支持 summary_plot，需用matplotlib）
plt.savefig("../results/figs/XBG_withnan_shap2.png", dpi=300, bbox_inches='tight')
plt.close()